# 03 — PyTorch MNIST：訓練 → 部署 → 推論（UI 為主）

本 notebook 為 **操作指引**。下列步驟優先在 **OpenShift AI Dashboard** / **OpenShift Console** 完成，
不在 Notebook Python 內提交 TrainJob 或呼叫推論 API。

**產出位置**（PVC `model-storage`）：
```
mnist-onnx/
├── model.onnx
└── mnist_model.pth
```

**Workbench 映像**：PyTorch 或 CUDA **3.4**

## 步驟 0：建立 Cluster storage（OpenShift AI Dashboard）

Projects → `rhoai-quickstart` → **Settings** → **Cluster storage** → **Create cluster storage**

| 欄位 | 值 |
|------|-----|
| Name | `model-storage` |
| Size | `10 GiB` |
| Access mode | **ReadWriteOnce (RWO)** |

**或 OpenShift Console**：Workloads → **PersistentVolumeClaims** → Create

> `WaitForFirstConsumer` 時 Pending 正常；TrainJob 掛載後變 **Bound**。

## 步驟 1：提交 TrainJob（CLI / Console，非 Notebook）

YAML 在 repo：
- `training/pytorch-mnist/k8s/configmap.yaml`
- `training/pytorch-mnist/k8s/trainjob.yaml`

**方式 A — 本機 Terminal**（repo 根目錄）：
```bash
export NAMESPACE=rhoai-quickstart
oc apply -f training/pytorch-mnist/k8s/configmap.yaml -n "$NAMESPACE"
oc apply -f training/pytorch-mnist/k8s/trainjob.yaml -n "$NAMESPACE"
```

**方式 B — OpenShift Console**：
1. 切換至專案 `rhoai-quickstart`
2. 右上角 **+** → **Import YAML**
3. 依序 Import：`configmap.yaml`、`trainjob.yaml`

**方式 C — 部署腳本**：`./scripts/deploy-training.sh pytorch`

詳見 [training/pytorch-mnist/README.md](../../training/pytorch-mnist/README.md)

## 步驟 2：監控訓練（OpenShift AI Dashboard）

1. Dashboard → **Develop & train** → **Training jobs**
2. 找到 **`pytorch-mnist-job`**，等待 **Complete** / **Succeeded**
3. 點進 job 查看 Pod 日誌

**或 OpenShift Console**：Workloads → Pods → 篩選 `pytorch-mnist-job` → **Logs**

成功後 `model-storage/mnist-onnx/` 應有 `model.onnx`。

## 步驟 3：部署 ONNX 模型（OpenShift AI Dashboard）

Dashboard → **Deploy model**

| 欄位 | 值 |
|------|-----|
| Model deployment name | `mnist-classifier` |
| Model type | **Predictive** |
| Model framework | **ONNX** |
| Source model location | **Existing cluster storage** |
| Cluster storage | `model-storage` |
| Model path | `mnist-onnx/` |
| Deployment mode | **Standard** |

等待 Status **Ready**。

**或 CLI**（選用）：
```bash
oc apply -f inference/pytorch-mnist/k8s/inferenceservice.yaml -n rhoai-quickstart
```

## 步驟 4：確認推論部署（Dashboard）

Dashboard → **Models** → **`mnist-classifier`** → Status **Ready**

叢集內 URL：
```
http://mnist-classifier-predictor.rhoai-quickstart.svc.cluster.local:8080
```

## 步驟 5：Terminal 測試推論

MNIST 輸入為 784 維，不適合手打 curl JSON。

**方式 A** — repo 測試腳本（已 git clone 時）：
```bash
cd /opt/app-root/src/rhoai-quickstart
python inference/pytorch-mnist/clients/infer.py --namespace rhoai-quickstart
```

**方式 B** — 執行下一格 smoke test（stdlib，無需 pip install）。

In [ ]:
%%bash
set -euo pipefail
export NAMESPACE="${NB_NAMESPACE:-rhoai-quickstart}"
export MODEL="mnist-classifier"
export URL="http://${MODEL}-predictor.${NAMESPACE}.svc.cluster.local:8080"
export TOKEN=$(oc whoami -t 2>/dev/null || cat /var/run/secrets/kubernetes.io/serviceaccount/token)

python3 - <<'EOF'
import json, os, urllib.request

namespace = os.environ["NAMESPACE"]
model = os.environ["MODEL"]
url = os.environ["URL"]
token = os.environ["TOKEN"]

payload = {
    "inputs": [{
        "name": "input",
        "shape": [1, 1, 28, 28],
        "datatype": "FP32",
        "data": [0.0] * 784,
    }]
}
req = urllib.request.Request(
    f"{url.rstrip('/')}/v2/models/{model}/infer",
    data=json.dumps(payload).encode(),
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req) as resp:
    result = json.loads(resp.read())
logits = result["outputs"][0]["data"]
pred = max(range(len(logits)), key=lambda i: logits[i])
print(f"URL: {url}")
print(f"Predicted digit: {pred}")
print(json.dumps(result, indent=2))
EOF

---

### 選用：事後 .pth → ONNX 轉換（CLI 教學）

若需示範「已有 PyTorch 權重再轉 ONNX」：
```bash
oc apply -f inference/pytorch-mnist/k8s/export-onnx-job.yaml -n rhoai-quickstart
```
見 [inference/pytorch-mnist/README.md](../../inference/pytorch-mnist/README.md)